# 16 - DRM box: reused vs interpolated Green's functions

A DRM box turns the difference between its two layers of nodes (one element apart) into effective forces, so
neighbouring nodes must stay coherent. This notebook drives a small DRM box (2.5 m elements, 531 nodes) with a
2 x 2 patch of point sources at about 2 km depth in a 4-layer crust and compares three ways of getting the Green's
functions:

| run | engine | Green's functions |
|---|---|---|
| exact | `run_nearest`, tolerances 1 cm, one wavenumber step per pair of depths | one per source-receiver pair |
| reuse | `run_nearest`, tolerances 40 m / 5 m / 200 m | one per slot, shared by every pair inside it |
| inter | `run_inter(scheme='tensor')`, 10 Hz band | depths exact, interpolated in distance |

The model, the engines and the metric come from `../drm_coherence.py`. Running the three engines takes about
10 minutes on one process (the exact reference dominates; `run_inter` alone takes a few seconds).

In [1]:
import os, sys, tempfile, time
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))
import drm_coherence as dc

dt = dc.FK["dt"]
work = tempfile.mkdtemp(prefix="run_inter_nb_")


    ▄████████     ▄█    █▄     ▄████████     ▄█   ▄█▄    ▄████████    ▄████████
    ███    ███   ███    ███    ███    ███    ███ ▄███▀   ███    ███   ███    ███
    ███    █▀    ███    ███    ███    ███    ███▐██▀     ███    █▀    ███    ███
    ███         ▄███▄▄▄▄███▄▄  ███    ███   ▄█████▀     ▄███▄▄▄      ▄███▄▄▄▄██▀
  ▀███████████  ▀███▀▀▀▀███▀  ▀███████████  ▀█████▄    ▀▀███▀▀▀     ▀▀███▀▀▀▀▀
           ███   ███    ███    ███    ███    ███▐██▄     ███    █▄   ▀███████████
     ▄█    ███   ███    ███    ███    ███    ███ ▀███▄   ███    ███   ███    ███
   ▄████████▀    ███    █▀     ███    █▀     ███   ▀█▀   ██████████   ███    ███
                                                                      ███    ███

         ▄▄▄▄███▄▄▄▄      ▄████████    ▄█   ▄█▄    ▄████████    ▄████████
       ▄██▀▀▀███▀▀▀██▄   ███    ███    ███ ▄███▀   ███    ███   ███    ███
       ███   ███   ███   ███    ███    ███▐██▀     ███    █▀    ███    ███
       ███   ███   ███   ███    ███   ▄█████▀  

## Model: crust, sources and DRM box

In [2]:
crust, fault, box = dc.crust(), dc.fault(), dc.box()
xyz = np.array([box.get_station_by_id(i).x for i in range(box.nstations)])
inner = np.array([box.get_station_by_id(i).is_internal for i in range(box.nstations)])
src = np.array([s.x for s in fault])

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
top = 0.0
for d, vp, vs in [(0.2, 1.32, 0.75), (0.8, 2.75, 1.57), (14.5, 5.50, 3.14), (0.0, 7.00, 4.00)]:
    bot = top + d if d > 0 else 3.0
    ax[0].plot([vs, vs], [top, bot], color="tab:red", lw=2)
    ax[0].plot([vp, vp], [top, bot], color="tab:blue", lw=2)
    top = bot
ax[0].plot([], [], color="tab:red", label="Vs"); ax[0].plot([], [], color="tab:blue", label="Vp")
ax[0].scatter(np.zeros(len(src)) + 0.3, src[:, 2], marker="*", s=80, color="k", label="sources")
ax[0].set_ylim(3.0, 0.0); ax[0].set_xlabel("velocity (km/s)"); ax[0].set_ylabel("depth (km)"); ax[0].legend(fontsize=8)
ax[0].set_title("crust (top 3 km)")

ax[1].scatter(src[:, 1], src[:, 0], marker="*", s=80, color="k", label="sources")
ax[1].scatter(xyz[:, 1].mean(), xyz[:, 0].mean(), marker="s", s=40, color="tab:red", label="DRM box")
ax[1].set_xlabel("east (km)"); ax[1].set_ylabel("north (km)"); ax[1].set_aspect("equal"); ax[1].legend(fontsize=8)
ax[1].set_title("plan view")

m = np.isclose(xyz[:, 1], np.median(xyz[:, 1]))
ax[2].scatter(1000 * xyz[m & ~inner, 0], 1000 * xyz[m & ~inner, 2], s=14, color="#7f9fc6", label="exterior layer")
ax[2].scatter(1000 * xyz[m & inner, 0], 1000 * xyz[m & inner, 2], s=14, color="tab:red", label="interior layer")
ax[2].invert_yaxis(); ax[2].set_xlabel("north (m)"); ax[2].set_ylabel("depth (m)"); ax[2].set_aspect("equal")
ax[2].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1.0)); ax[2].set_title("box section (one plane)")
fig.tight_layout()
fig.savefig("model_and_box.png", dpi=150, bbox_inches="tight")
print(f"{box.nstations} nodes ({inner.sum()} interior), {len(src)} sources at {src[:, 2].min():.2f}-{src[:, 2].max():.2f} km")

531 nodes (210 interior), 4 sources at 1.92-2.08 km


## Run the three engines

In [3]:
res, wall = {}, {}
for kind in ("exact", "reuse", "inter"):
    t0 = time.perf_counter()
    res[kind] = dc.run(kind, os.path.join(work, kind, "gf"))[0]
    wall[kind] = time.perf_counter() - t0
    print(f"{kind:5s} {wall[kind]:7.1f} s")
ref = res["exact"]



ShakerMaker run_nearest | stage=all | dt=0.005 nfft=1024 dk=0.1 tb=100 tmin=0.0 tmax=4.0
----------------------------------------------------------------------------------------
Hybrid Parallelization:
   MPI processes  : 1
   OpenMP threads : 1
   Total threads  : 1 x 1 = 1
   DB file        : /tmp/run_inter_nb_gukb7ez4/exact/gf
----------------------------------------------------------------------------------------


ShakerMaker Gen GF database pairs begin (one process). delta_h=1e-05 delta_v_rec=1e-05 delta_v_src=1e-05
  Stations    : 531
  Sources     : 4
  Total pairs : 2124



Need only 2120 pairs of 2124 (99.8% of total, 0.2% reduction)
Stage 0 done. Time: 1.4s
Mapping database written to: /tmp/run_inter_nb_gukb7ez4/exact/gf_map.h5 (0.0s)


ShakerMaker Gen Green's functions database begin. dt=0.005 nfft=1024 dk=0.1 tb=100
----------------------------------------------------------------------------------
  MPI processes  : 1
  OpenMP threads : 1
  Loading database: /tmp/run_inter_nb_gukb7ez4/exact/gf
  Slots to compute: 2120
  Map file : /tmp/run_inter_nb_gukb7ez4/exact/gf_map.h5
  GF  file : /tmp/run_inter_nb_gukb7ez4/exact/gf_gf.h5




ShakerMaker Generate GF database done. Total time: 510.03 s
--------------------------------------------------


ShakerMaker Run (Stage 2 - OP) begin. dt=0.005 nfft=1024 dk=0.1 tb=100 tmin=0.0 tmax=4.0
----------------------------------------------------------------------------------------
  MPI processes  : 1
  OpenMP threads : 1
  Loading database: /tmp/run_inter_nb_gukb7ez4/exact/gf
  writer_mode     : progressive
  Map file : /tmp/run_inter_nb_gukb7ez4/exact/gf_map.h5
  GF  file : /tmp/run_inter_nb_gukb7ez4/exact/gf_gf.h5
  pair_to_slot: O(1) lookup (531 stations x 4 sources)
  t0 check : 2120 slots, max |t0 - /t0| = 0 s, 0 above dt/10
  rank=0 sta 1/531 (0.2%)  sta_time=0.0s  ETA_total=0:00:02.1
  rank=0 sta 2/531 (0.4%)  sta_time=0.0s  ETA_total=0:00:01.8
  rank=0 sta 3/531 (0.6%)  sta_time=0.0s  ETA_total=0:00:01.8
  rank=0 sta 4/531 (0.8%)  sta_time=0.0s  ETA_total=0:00:01.8
  rank=0 sta 5/531 (0.9%)  sta_time=0.0s  ETA_total=0:00:01.8
  rank=0 sta 6/531 (1.1%)  sta_time=0.0s

  rank=0 sta 35/531 (6.6%)  sta_time=0.0s  ETA_total=0:00:01.7
  rank=0 sta 36/531 (6.8%)  sta_time=0.0s  ETA_total=0:00:01.6
  rank=0 sta 37/531 (7.0%)  sta_time=0.0s  ETA_total=0:00:01.7
  rank=0 sta 38/531 (7.2%)  sta_time=0.0s  ETA_total=0:00:01.6
  rank=0 sta 39/531 (7.3%)  sta_time=0.0s  ETA_total=0:00:01.6
  rank=0 sta 40/531 (7.5%)  sta_time=0.0s  ETA_total=0:00:01.6
  rank=0 sta 41/531 (7.7%)  sta_time=0.0s  ETA_total=0:00:01.6
  rank=0 sta 42/531 (7.9%)  sta_time=0.0s  ETA_total=0:00:01.7
  rank=0 sta 43/531 (8.1%)  sta_time=0.0s  ETA_total=0:00:01.6
  rank=0 sta 44/531 (8.3%)  sta_time=0.0s  ETA_total=0:00:01.6
  rank=0 sta 45/531 (8.5%)  sta_time=0.0s  ETA_total=0:00:01.6
  rank=0 sta 46/531 (8.7%)  sta_time=0.0s  ETA_total=0:00:01.7
  rank=0 sta 47/531 (8.9%)  sta_time=0.0s  ETA_total=0:00:01.7
  rank=0 sta 48/531 (9.0%)  sta_time=0.0s  ETA_total=0:00:01.6
  rank=0 sta 49/531 (9.2%)  sta_time=0.0s  ETA_total=0:00:01.7
  rank=0 sta 50/531 (9.4%)  sta_time=0.0s  ETA_total=0:

  rank=0 sta 95/531 (17.9%)  sta_time=0.0s  ETA_total=0:00:01.6
  rank=0 sta 96/531 (18.1%)  sta_time=0.0s  ETA_total=0:00:01.4
  rank=0 sta 97/531 (18.3%)  sta_time=0.0s  ETA_total=0:00:01.4
  rank=0 sta 98/531 (18.5%)  sta_time=0.0s  ETA_total=0:00:01.4
  rank=0 sta 99/531 (18.6%)  sta_time=0.0s  ETA_total=0:00:01.5
  rank=0 sta 100/531 (18.8%)  sta_time=0.0s  ETA_total=0:00:01.4
  rank=0 sta 101/531 (19.0%)  sta_time=0.0s  ETA_total=0:00:01.4
  rank=0 sta 102/531 (19.2%)  sta_time=0.0s  ETA_total=0:00:01.4
  rank=0 sta 103/531 (19.4%)  sta_time=0.0s  ETA_total=0:00:01.4
  rank=0 sta 104/531 (19.6%)  sta_time=0.0s  ETA_total=0:00:01.4
  rank=0 sta 105/531 (19.8%)  sta_time=0.0s  ETA_total=0:00:01.4
  rank=0 sta 106/531 (20.0%)  sta_time=0.0s  ETA_total=0:00:01.4
  rank=0 sta 107/531 (20.2%)  sta_time=0.0s  ETA_total=0:00:01.4
  rank=0 sta 108/531 (20.3%)  sta_time=0.0s  ETA_total=0:00:01.4
  rank=0 sta 109/531 (20.5%)  sta_time=0.0s  ETA_total=0:00:01.4
  rank=0 sta 110/531 (20.7%)  

  rank=0 sta 155/531 (29.2%)  sta_time=0.0s  ETA_total=0:00:01.3
  rank=0 sta 156/531 (29.4%)  sta_time=0.0s  ETA_total=0:00:01.3
  rank=0 sta 157/531 (29.6%)  sta_time=0.0s  ETA_total=0:00:01.3
  rank=0 sta 158/531 (29.8%)  sta_time=0.0s  ETA_total=0:00:01.3
  rank=0 sta 159/531 (29.9%)  sta_time=0.0s  ETA_total=0:00:01.3
  rank=0 sta 160/531 (30.1%)  sta_time=0.0s  ETA_total=0:00:01.3
  rank=0 sta 161/531 (30.3%)  sta_time=0.0s  ETA_total=0:00:01.3
  rank=0 sta 162/531 (30.5%)  sta_time=0.0s  ETA_total=0:00:01.3
  rank=0 sta 163/531 (30.7%)  sta_time=0.0s  ETA_total=0:00:01.3
  rank=0 sta 164/531 (30.9%)  sta_time=0.0s  ETA_total=0:00:01.2
  rank=0 sta 165/531 (31.1%)  sta_time=0.0s  ETA_total=0:00:01.2
  rank=0 sta 166/531 (31.3%)  sta_time=0.0s  ETA_total=0:00:01.3
  rank=0 sta 167/531 (31.5%)  sta_time=0.0s  ETA_total=0:00:01.2
  rank=0 sta 168/531 (31.6%)  sta_time=0.0s  ETA_total=0:00:01.2
  rank=0 sta 169/531 (31.8%)  sta_time=0.0s  ETA_total=0:00:01.2
  rank=0 sta 170/531 (32.

... (13 outputs left out)

  rank=0 sta 392/531 (73.8%)  sta_time=0.0s  ETA_total=0:00:00.4
  rank=0 sta 393/531 (74.0%)  sta_time=0.0s  ETA_total=0:00:00.4
  rank=0 sta 394/531 (74.2%)  sta_time=0.0s  ETA_total=0:00:00.5
  rank=0 sta 395/531 (74.4%)  sta_time=0.0s  ETA_total=0:00:00.4
  rank=0 sta 396/531 (74.6%)  sta_time=0.0s  ETA_total=0:00:00.4
  rank=0 sta 397/531 (74.8%)  sta_time=0.0s  ETA_total=0:00:00.4
  rank=0 sta 398/531 (75.0%)  sta_time=0.0s  ETA_total=0:00:00.4
  rank=0 sta 399/531 (75.1%)  sta_time=0.0s  ETA_total=0:00:00.4
  rank=0 sta 400/531 (75.3%)  sta_time=0.0s  ETA_total=0:00:00.4
  rank=0 sta 401/531 (75.5%)  sta_time=0.0s  ETA_total=0:00:00.4
  rank=0 sta 402/531 (75.7%)  sta_time=0.0s  ETA_total=0:00:00.4
  rank=0 sta 403/531 (75.9%)  sta_time=0.0s  ETA_total=0:00:00.4
  rank=0 sta 404/531 (76.1%)  sta_time=0.0s  ETA_total=0:00:00.4
  rank=0 sta 405/531 (76.3%)  sta_time=0.0s  ETA_total=0:00:00.4
  rank=0 sta 406/531 (76.5%)  sta_time=0.0s  ETA_total=0:00:00.4
  rank=0 sta 407/531 (76.

  rank=0 sta 461/531 (86.8%)  sta_time=0.0s  ETA_total=0:00:00.2
  rank=0 sta 462/531 (87.0%)  sta_time=0.0s  ETA_total=0:00:00.2
  rank=0 sta 463/531 (87.2%)  sta_time=0.0s  ETA_total=0:00:00.2
  rank=0 sta 464/531 (87.4%)  sta_time=0.0s  ETA_total=0:00:00.2
  rank=0 sta 465/531 (87.6%)  sta_time=0.0s  ETA_total=0:00:00.2
  rank=0 sta 466/531 (87.8%)  sta_time=0.0s  ETA_total=0:00:00.2
  rank=0 sta 467/531 (87.9%)  sta_time=0.0s  ETA_total=0:00:00.2
  rank=0 sta 468/531 (88.1%)  sta_time=0.0s  ETA_total=0:00:00.2
  rank=0 sta 469/531 (88.3%)  sta_time=0.0s  ETA_total=0:00:00.2
  rank=0 sta 470/531 (88.5%)  sta_time=0.0s  ETA_total=0:00:00.2
  rank=0 sta 471/531 (88.7%)  sta_time=0.0s  ETA_total=0:00:00.2
  rank=0 sta 472/531 (88.9%)  sta_time=0.0s  ETA_total=0:00:00.2
  rank=0 sta 473/531 (89.1%)  sta_time=0.0s  ETA_total=0:00:00.2
  rank=0 sta 474/531 (89.3%)  sta_time=0.0s  ETA_total=0:00:00.2
  rank=0 sta 475/531 (89.5%)  sta_time=0.0s  ETA_total=0:00:00.2
  rank=0 sta 476/531 (89.

  rank=0 sta 530/531 (99.8%)  sta_time=0.0s  ETA_total=0:00:00.0
  rank=0 sta 531/531 (100.0%)  sta_time=0.0s  ETA_total=0:00:00.0


ShakerMaker Run (Stage 2 - OP) done. Total time: 1.62 s
--------------------------------------------------

ALL STAGES COMPLETE
  Total time: 33.10 s

reuse    33.1 s


inter     3.6 s


## Error of the DRM layer difference, per band

For every interior node, the difference with its nearest exterior node is compared with the same difference in
the exact run (median over the nodes of the relative error). This is the quantity the DRM turns into forces.

In [4]:
bands = [(0.1, 2.5), (2.5, 5.0), (5.0, 7.5), (7.5, 10.0)]
err = {k: [dc.layer_difference_error(res[k], ref, box, *b) for b in bands] for k in ("reuse", "inter")}
for k in err:
    print(f"{k:5s} " + "  ".join(f"{a:g}-{b:g} Hz {100 * e:7.2f} %" for (a, b), e in zip(bands, err[k])))

x, w = np.arange(len(bands)), 0.38
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(x - w / 2, [100 * e for e in err["reuse"]], w, color="#7f9fc6", label="reuse (run_nearest 40 m / 5 m / 200 m)")
ax.bar(x + w / 2, [100 * e for e in err["inter"]], w, color="#c0392b", label="interpolated (run_inter, tensor)")
ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels([f"{a:g}-{b:g} Hz" for a, b in bands])
ax.set_ylabel("median error of the\nlayer difference (%)"); ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig("layer_difference_error.png", dpi=150, bbox_inches="tight")

reuse 0.1-2.5 Hz   48.55 %  2.5-5 Hz   39.91 %  5-7.5 Hz   34.72 %  7.5-10 Hz   49.76 %
inter 0.1-2.5 Hz    0.44 %  2.5-5 Hz    0.14 %  5-7.5 Hz    0.10 %  7.5-10 Hz    0.20 %


## One pair of nodes

An interior node and its nearest exterior node, 2.5 m apart. The traces of the two nodes are nearly identical, so
the error only shows in their difference: with reuse both nodes can fall in the same slot (identical traces,
zero difference) or in neighbouring slots (a jump).

In [5]:
ii, oo = np.flatnonzero(inner), np.flatnonzero(~inner)
pair = oo[np.argmin(np.linalg.norm(xyz[ii][:, None, :] - xyz[oo][None, :, :], axis=2), axis=1)]
e_reuse = np.array([np.linalg.norm((res["reuse"][i] - res["reuse"][o]) - (ref[i] - ref[o])) / np.linalg.norm(ref[i] - ref[o])
                    for i, o in zip(ii, pair)])
k = int(np.argsort(e_reuse)[len(e_reuse) // 2])          # a node with the median reuse error
i, o = ii[k], pair[k]
t = np.arange(ref.shape[-1]) * dt

fig, ax = plt.subplots(3, 2, figsize=(12, 6.5), sharex=True)
for j, comp in enumerate("ZEN"):
    ax[j, 0].plot(t, ref[i, j], color="k", lw=0.9, label="interior node")
    ax[j, 0].plot(t, ref[o, j], "--", color="#7f9fc6", lw=1.2, label="exterior node")
    ax[j, 1].plot(t, ref[i, j] - ref[o, j], color="k", lw=1.4, label="exact")
    ax[j, 1].plot(t, res["reuse"][i, j] - res["reuse"][o, j], color="#7f9fc6", lw=1.0, label="reuse")
    ax[j, 1].plot(t, res["inter"][i, j] - res["inter"][o, j], "--", color="#c0392b", lw=1.0, label="interpolated")
    ax[j, 0].set_ylabel(comp)
ax[0, 0].set_title("exact traces of the two nodes"); ax[0, 1].set_title("difference interior - exterior")
ax[0, 0].legend(fontsize=7); ax[0, 1].legend(fontsize=7)
ax[-1, 0].set_xlabel("time (s)"); ax[-1, 1].set_xlabel("time (s)")
fig.tight_layout()
fig.savefig("node_pair.png", dpi=150, bbox_inches="tight")
print(f"interior node {xyz[i] * 1000} m, exterior node {xyz[o] * 1000} m")

interior node [1507.5 1490.     2.5] m, exterior node [1507.5 1487.5    2.5] m


## Spectrum of the error of the layer difference (median over the nodes)

In [6]:
nt = ref.shape[-1]
f = np.fft.rfftfreq(nt, dt)
d_ref = np.fft.rfft(ref[ii] - ref[pair], axis=-1)
amp = np.median(np.abs(d_ref), axis=(0, 1))

fig, ax = plt.subplots(figsize=(7, 3.2))
for kind, color, ls in (("reuse", "#7f9fc6", "-"), ("inter", "#c0392b", "-")):
    d = np.fft.rfft(res[kind][ii] - res[kind][pair], axis=-1)
    ax.semilogy(f, np.median(np.abs(d - d_ref), axis=(0, 1)) / amp.max(), color=color, ls=ls, label=kind)
ax.semilogy(f, amp / amp.max(), color="k", lw=0.8, label="exact difference (scale)")
ax.axvspan(10.0, f.max(), color="0.9", zorder=0)
ax.set_xlim(0, 15); ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("|error| (normalised)"); ax.legend(fontsize=8)
ax.set_title("band of use: 0-10 Hz (shaded: above the band)")
fig.tight_layout()
fig.savefig("error_spectrum.png", dpi=150, bbox_inches="tight")

## Reading

- **Reuse breaks the box.** Sharing one Green's function per slot makes neighbouring nodes identical or makes them
  jump at the slot edge; the layer difference is wrong by tens of percent in every band, and the DRM turns that
  error into spurious forces. Smaller tolerances move the slot edges, they do not remove them.
- **Interpolation keeps it.** With the depths computed exactly and the distance interpolated in reduced time, the
  layer difference stays within a fraction of a percent of the exact one in the whole 10 Hz band.
- **Cost.** `run_inter` takes seconds here; the exact reference is the expensive run.

Method and limits (near field, band of use): `docs/web/background/gf_interpolation.md`; use:
`docs/web/guides/run_inter.md`.